# bandaid.io — test notebook

Test the IO functions against two real session folders:
1. **Multi-file session** — `AMD4/AMD4_20260211` (3 tiffs, power stepping)
2. **Single-file session** — `AMD8/AMD8_20260618/AMD8_20260618_002` (1 tiff, 13K frames)

In [4]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))     
from bandaid.io import (
    read_scanimage_metadata,
    save_session_metadata,
    load_session_metadata,
    load_tiff_stack,
    get_files,
)

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [6]:
session_multi  = Path("/Users/hakan/Downloads/AMD4/AMD4_20260211")
session_single = Path("/Users/hakan/Downloads/AMD8/AMD8_20260618/AMD8_20260618_002")

## 1. read_scanimage_metadata — single file

In [7]:
tiff_path = sorted(session_multi.glob("*.tif"))[0]
meta = read_scanimage_metadata(tiff_path)

print(f"File: {tiff_path.name}")
print(f"Shape: {meta['shape']}  dtype: {meta['dtype']}  n_frames: {meta['n_frames']}")
print(f"\nSI params: {len(meta['si_params'])} keys")
print(f"ROI: {'present' if meta['roi'] else 'missing'}")
print(f"Frame metadata: {len(meta['frame_meta'])} frames")

File: AMD4_20260211_00002.tif
Shape: (256, 256)  dtype: int16  n_frames: 450

SI params: 334 keys
ROI: present
Frame metadata: 450 frames


In [9]:
# Check a few key SI params
si = meta['si_params']
for key in [
    'SI.hRoiManager.scanFrameRate',
    'SI.hRoiManager.pixelsPerLine',
    'SI.hRoiManager.linesPerFrame',
    'SI.hRoiManager.scanZoomFactor',
    'SI.hScan2D.bidirectional',
    'SI.hScan2D.scannerType',
    'SI.hBeams.powers',
    'SI.hChannels.channelSave',
]:
    print(f"  {key} = {si.get(key, 'MISSING')}")

  SI.hRoiManager.scanFrameRate = 14.4927536231884
  SI.hRoiManager.pixelsPerLine = 256
  SI.hRoiManager.linesPerFrame = 256
  SI.hRoiManager.scanZoomFactor = 1.8
  SI.hScan2D.bidirectional = True
  SI.hScan2D.scannerType = RG
  SI.hBeams.powers = 20
  SI.hChannels.channelSave = 1


In [10]:
# Check per-frame metadata
fm0 = meta['frame_meta'][0]
fm_last = meta['frame_meta'][-1]

print("First frame:")
for k, v in fm0.items():
    print(f"  {k} = {v}")

print(f"\nLast frame:")
for k, v in fm_last.items():
    print(f"  {k} = {v}")

First frame:
  frameNumbers = 1
  acquisitionNumbers = 1
  frameNumberAcquisition = 1
  frameTimestamps_sec = 0.0
  acqTriggerTimestamps_sec = -0.00045539
  nextFileMarkerTimestamps_sec = -1.0
  endOfAcquisition = 0
  endOfAcquisitionMode = 0
  dcOverVoltage = 0
  epoch = [2026  2 11 13 18 59.046]
  auxTrigger0 = []
  auxTrigger1 = []
  auxTrigger2 = []
  auxTrigger3 = []
  I2CData = {}

Last frame:
  frameNumbers = 450
  acquisitionNumbers = 1
  frameNumberAcquisition = 450
  frameTimestamps_sec = 30.981000005
  acqTriggerTimestamps_sec = -1.0
  nextFileMarkerTimestamps_sec = -1.0
  endOfAcquisition = 1
  endOfAcquisitionMode = 1
  dcOverVoltage = 0
  epoch = [2026  2 11 13 18 59.046]
  auxTrigger0 = []
  auxTrigger1 = []
  auxTrigger2 = []
  auxTrigger3 = []
  I2CData = {}


## 2. save_session_metadata — multi-file session

AMD4_20260211 has 3 tiffs with power stepping (20→30→40%). Expect a warning about differing params.

In [11]:
import warnings
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    json_path = save_session_metadata(session_multi)
    if w:
        for warning in w:
            print(f"WARNING: {warning.message}")

Session metadata saved to: /Users/hakan/Downloads/AMD4/AMD4_20260211/session_metadata.json (91.5 KB)


In [ ]:
# Load it back and inspect
sm = load_session_metadata(json_path)

print("session:", sm['session'])
print()
print("acquisition:")
for k, v in sm['acquisition'].items():
    print(f"  {k}: {v}")

In [ ]:
# Check per-file entries
for f in sm['files']:
    ts = f['frames']['timestamps_sec']
    fna = f['frames']['frame_numbers_acquisition']
    print(f"{f['filename']}: {f['n_frames']} frames, "
          f"acq_complete={f['acquisition_complete']}, "
          f"shape={f['shape']}, dtype={f['dtype']}, "
          f"t=[{ts[0]:.3f} .. {ts[-1]:.3f}]s, "
          f"frame_acq=[{fna[0]}..{fna[-1]}]")

print(f"\nTotal frames across session: {sm['session']['total_frames']}")

## 3. save_session_metadata — single-file session

AMD8_20260618_002 has one 13K-frame tiff. No warnings expected.

In [14]:
json_path_single = save_session_metadata(session_single)

Session metadata saved to: /Users/hakan/Downloads/AMD8/AMD8_20260618/AMD8_20260618_002/session_metadata.json (878.3 KB)


In [ ]:
ss = load_session_metadata(json_path_single)

print("session:", ss['session'])
print()
print("acquisition:")
for k, v in ss['acquisition'].items():
    print(f"  {k}: {v}")
print()
f = ss['files'][0]
ts = f['frames']['timestamps_sec']
fna = f['frames']['frame_numbers_acquisition']
print(f"{f['filename']}: {f['n_frames']} frames, "
      f"acq_complete={f['acquisition_complete']}, "
      f"shape={f['shape']}, dtype={f['dtype']}, "
      f"t=[{ts[0]:.3f} .. {ts[-1]:.3f}]s, "
      f"frame_acq=[{fna[0]}..{fna[-1]}]")
print(f"\nTotal frames: {ss['session']['total_frames']}")

## 4. load_session_metadata — directory shortcut

Can also pass the session directory instead of the JSON path.

In [16]:
# Loading by directory (finds session_metadata.json automatically)
sm2 = load_session_metadata(session_multi)
assert sm2 == sm, "Round-trip mismatch!"
print("Round-trip OK: load by directory matches load by path")

Round-trip OK: load by directory matches load by path


## 5. Quick sanity checks

In [ ]:
# Sanity checks
from tifffile import TiffFile

for session_dir, session_meta in [(session_multi, sm), (session_single, ss)]:
    print(f"\n--- {session_dir.name} (total_frames={session_meta['session']['total_frames']}) ---")
    for finfo in session_meta['files']:
        tp = session_dir / finfo['filename']
        with TiffFile(str(tp)) as tif:
            actual_frames = len(tif.pages)

        # Frame count vs actual pages
        count_ok = actual_frames == finfo['n_frames']
        
        # frameNumberAcquisition should match page count
        fna = finfo['frames']['frame_numbers_acquisition']
        fna_ok = fna[-1] == actual_frames if fna else False
        
        # Timestamps monotonic
        ts = finfo['frames']['timestamps_sec']
        ts_mono = all(ts[i] <= ts[i+1] for i in range(len(ts)-1))
        
        status = "OK" if (count_ok and fna_ok and ts_mono) else "ISSUE"
        print(f"  {finfo['filename']}: pages={actual_frames} n_frames={finfo['n_frames']} "
              f"last_fna={fna[-1] if fna else '?'} "
              f"acq_complete={finfo['acquisition_complete']} "
              f"ts_mono={ts_mono} [{status}]")

print("\nAll checks passed!")